In [1]:
print("Here begins the hybrid models notebook.")

Here begins the hybrid models notebook.


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/world_happiness_ml.csv")

df_ml.head()

,Country,Year,Climate_Index,Corruption_Perception,Crime_Rate,Education_Index,Employment_Rate,Freedom,GDP_per_Capita,Generosity,Happiness_Score,Healthy_Life_Expectancy,Income_Inequality,Internet_Access,Life_Satisfaction,Mental_Health_Index,Political_Stability,Population,Public_Health_Expenditure,Public_Trust,Social_Support,Unemployment_Rate,Urbanization_Rate,Work_Life_Balance,Year_bin
0,Australia,2005,64.565333,0.517333,44.341333,0.778000,72.108667,0.372000,28886.978000,0.181333,5.430667,68.128667,41.165333,71.750667,5.685333,74.255333,0.552667,7.037067e+08,5.525333,0.512667,0.528667,11.041333,59.478667,6.286667,2005
1,Australia,2006,69.659444,0.449444,47.636111,0.736667,73.490556,0.495000,28936.431111,0.100556,6.030556,63.465000,36.857222,67.400556,6.388333,75.430000,0.516111,7.845685e+08,5.838333,0.481111,0.677222,8.176111,61.441111,5.902222,2005
2,Australia,2007,66.739091,0.487273,46.938636,0.780455,73.953636,0.529545,36317.625455,0.167273,5.600000,65.911818,38.910000,68.869545,6.788636,71.362727,0.564545,6.854900e+08,6.015000,0.530000,0.505000,9.251818,64.544545,5.937273,2005
3,Australia,2008,59.651111,0.480556,47.990556,0.696667,69.658333,0.616667,27944.319444,0.170000,5.868889,69.021111,33.376667,73.648889,6.635556,60.281667,0.658333,8.398988e+08,6.148333,0.530000,0.467778,13.337222,64.130556,5.455000,2005
4,Australia,2009,55.751905,0.549048,37.245238,0.756667,73.119048,0.574762,29764.803810,0.088571,5.182857,69.346667,47.194286,69.601905,6.563333,66.839048,0.486190,6.325921e+08,6.183810,0.436667,0.524762,11.704762,60.689524,6.234286,2005


In [8]:
# Inspect data structure
print(df_ml.shape)
print(df_ml.dtypes)

(200, 25)
Country                          str
Year                           int64
Climate_Index                float64
Corruption_Perception        float64
Crime_Rate                   float64
Education_Index              float64
Employment_Rate              float64
Freedom                      float64
GDP_per_Capita               float64
Generosity                   float64
Happiness_Score              float64
Healthy_Life_Expectancy      float64
Income_Inequality            float64
Internet_Access              float64
Life_Satisfaction            float64
Mental_Health_Index          float64
Political_Stability          float64
Population                   float64
Public_Health_Expenditure    float64
Public_Trust                 float64
Social_Support               float64
Unemployment_Rate            float64
Urbanization_Rate            float64
Work_Life_Balance            float64
Year_bin                       int64
dtype: object


In [9]:
# Define predictors and target
# - Life_Satisfaction is the regression target
# - Country is excluded because it is a high-cardinality identifier
# - Year_bin is excluded because it duplicates the information in Year
X = df_ml.drop(columns=["Life_Satisfaction", "Country", "Year_bin"])
y = df_ml["Life_Satisfaction"]

# Check shapes
print(X.shape, y.shape)

# Check missing values in predictors and target
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

(200, 22) (200,)
Missing values in X: 0
Missing values in y: 0


In [10]:
# Train/test split
# Since this is a regression task, stratification is not used
# The split is still useful for final model comparison, even though cross-validation
# is especially important for this small dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (160, 22)
Test shape: (40, 22)


In [11]:
# Optional: cross-validation baseline for Linear Regression
# With only around 200 rows, cross-validation gives a more stable estimate
# than relying only on a single train/test split

cv = KFold(n_splits=5, shuffle=True, random_state=42)

linreg_cv = LinearRegression()

cv_results_lin = cross_validate(
    linreg_cv,
    X, y,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

print("Linear Regression CV results")
print("Mean MAE:", -cv_results_lin["test_mae"].mean())
print("Mean RMSE:", -cv_results_lin["test_rmse"].mean())
print("Mean R2:", cv_results_lin["test_r2"].mean())

Linear Regression CV results
Mean MAE: 0.28394585871818984
Mean RMSE: 0.3572605839274692
Mean R2: -0.17171782397301874


In [12]:
# Hybrid 1: Random Forest feature selection
# Random Forest is used here as a nonlinear feature selector
# The final model will still be Linear Regression for interpretability

rf_selector = RandomForestRegressor(
    n_estimators=150,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 11

Selected features:
['Climate_Index', 'Corruption_Perception', 'Employment_Rate', 'Freedom', 'GDP_per_Capita', 'Generosity', 'Income_Inequality', 'Mental_Health_Index', 'Political_Stability', 'Population', 'Social_Support']


In [13]:
# Hybrid 1: Linear Regression on RF-selected features
# This combines nonlinear feature selection with a simple, interpretable regression model

linreg_hybrid_rf = LinearRegression()

linreg_hybrid_rf.fit(X_train_sel, y_train)
y_pred_hybrid1 = linreg_hybrid_rf.predict(X_test_sel)

print("Hybrid 1: RF Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid1)))
print("R2:", r2_score(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Linear Regression
MAE: 0.31741188557682287
RMSE: 0.39916208575834927
R2: -0.41984291426514275


In [14]:
# Coefficients for Hybrid 1
# These coefficients help interpret which selected predictors are most strongly
# associated with life satisfaction in the final linear model

coef_df_rf = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": linreg_hybrid_rf.coef_
})

coef_df_rf["Abs_Coefficient"] = coef_df_rf["Coefficient"].abs()
coef_df_rf.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
1,Corruption_Perception,7.958723e-01,7.958723e-01
3,Freedom,7.173005e-01,7.173005e-01
8,Political_Stability,7.063765e-01,7.063765e-01
5,Generosity,6.718671e-01,6.718671e-01
10,Social_Support,-1.853856e-01,1.853856e-01
2,Employment_Rate,1.583847e-02,1.583847e-02
7,Mental_Health_Index,-1.451488e-02,1.451488e-02
0,Climate_Index,-1.075006e-02,1.075006e-02
6,Income_Inequality,-5.200297e-03,5.200297e-03
4,GDP_per_Capita,7.049330e-06,7.049330e-06


In [15]:
# Hybrid 2: XGBoost feature selection
# XGBoost is used as an alternative nonlinear selector
# This allows comparison with Random Forest-based feature selection

xgb_selector = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector.fit(X_train, y_train)

selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

selector_xgb.fit(X_train, y_train)

X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

selected_features_xgb = X_train.columns[selector_xgb.get_support()]

print("Number of selected features:", len(selected_features_xgb))
print("\nSelected features:")
print(selected_features_xgb.tolist())

Number of selected features: 11

Selected features:
['Climate_Index', 'Corruption_Perception', 'Employment_Rate', 'Freedom', 'GDP_per_Capita', 'Happiness_Score', 'Internet_Access', 'Mental_Health_Index', 'Political_Stability', 'Public_Trust', 'Unemployment_Rate']


In [16]:
# Hybrid 2: Linear Regression on XGBoost-selected features
# This tests whether XGBoost-based feature selection produces better final
# linear predictions than Random Forest-based feature selection

linreg_hybrid_xgb = LinearRegression()

linreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)
y_pred_hybrid2 = linreg_hybrid_xgb.predict(X_test_sel_xgb)

print("Hybrid 2: XGB Feature Selection -> Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_hybrid2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hybrid2)))
print("R2:", r2_score(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Linear Regression
MAE: 0.3203243898031511
RMSE: 0.3981714302672319
R2: -0.41280402062244637


In [17]:
# Coefficients for Hybrid 2
# Again, coefficients are inspected after feature selection to interpret
# which variables remain important in the final linear model

coef_df_xgb = pd.DataFrame({
    "Feature": selected_features_xgb,
    "Coefficient": linreg_hybrid_xgb.coef_
})

coef_df_xgb["Abs_Coefficient"] = coef_df_xgb["Coefficient"].abs()
coef_df_xgb.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
3,Freedom,0.782482,0.782482
1,Corruption_Perception,0.696809,0.696809
8,Political_Stability,0.648127,0.648127
9,Public_Trust,0.204792,0.204792
5,Happiness_Score,0.041298,0.041298
2,Employment_Rate,0.016868,0.016868
7,Mental_Health_Index,-0.015288,0.015288
0,Climate_Index,-0.011616,0.011616
10,Unemployment_Rate,-0.011489,0.011489
6,Internet_Access,-0.004610,0.004610
